In [18]:
import os
from tqdm import tqdm
import polars as pl
from statsmodels.stats import multitest
from plotnine import *

## Genes2Keep

In [19]:
# Burden test genes

res_dir = '/home/dnanexus/data_dir/association_files/'
n_phenos = 127
burdens = "lofteeHC_mac20"
filename = f'regenie_{n_phenos}phenotypes_{burdens}.parquet'

!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/association_files/{filename} -o {res_dir}/{filename}

r = pl.read_parquet(f'{res_dir}/regenie_{n_phenos}phenotypes_{burdens}.parquet')
bt_genes = r.filter(pl.col('pval_fdr') < 0.05).sort('pval')['region'].unique().to_list()
len(bt_genes)

Error: path "/home/dnanexus/data_dir/association_files//regenie_127phenotypes_
lofteeHC_mac20.parquet" already exists but -f/--overwrite was not set


699

In [20]:
# Burden test genes
burdens = "lofteeHC_maf1e-3"
filename = f'regenie_{n_phenos}phenotypes_{burdens}.parquet'

!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/association_files/{filename} -o {res_dir}/{filename}

r = pl.read_parquet(f'{res_dir}/regenie_{n_phenos}phenotypes_{burdens}.parquet')
bt_genes_maf1e3 = r.filter(pl.col('pval_fdr') < 0.05).sort('pval')['region'].unique().to_list()

bt_genes = list(set(bt_genes_maf1e3).union(set(bt_genes)))
len(bt_genes)

Error: path "/home/dnanexus/data_dir/association_files//regenie_127phenotypes_
lofteeHC_maf1e-3.parquet" already exists but -f/--overwrite was not set


988

In [21]:
# Olink genes

!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/olink/preprocessed/proteomics_genes.txt

olink_genes = pl.read_csv('proteomics_genes.txt', has_header=False).select(pl.col('column_1').unique()).to_series().to_list()
len(olink_genes)

Error: path "/home/dnanexus/ukbgym/utils/proteomics_genes.txt" already exists
but -f/--overwrite was not set


2882

## Read Gencode file

In [22]:
!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/misc_data/gencode.v40.annotation.gtf.gz -o /home/dnanexus/data_dir/

Error: path "/home/dnanexus/data_dir/gencode.v40.annotation.gtf.gz" already
exists but -f/--overwrite was not set


In [23]:
gtf_path = "/home/dnanexus/data_dir/gencode.v40.annotation.gtf.gz"

gtf_pl = pl.read_csv(
    gtf_path,
    separator="\t",
    comment_prefix="#",
    has_header=False,
    new_columns=["Chromosome", "source", "Feature", "Start", "End", "score", "Strand", "frame", "attributes"]
)

gtf_pl = gtf_pl.with_row_index("row_nr")
gtf_pl

row_nr,Chromosome,source,Feature,Start,End,score,Strand,frame,attributes
u64,str,str,str,i64,i64,str,str,str,str
0,"""chr1""","""HAVANA""","""exon""",11869,12227,""".""","""+""",""".""","""gene_id ""ENSG00000223972.5""; t…"
1,"""chr1""","""HAVANA""","""gene""",11869,14409,""".""","""+""",""".""","""gene_id ""ENSG00000223972.5""; g…"
2,"""chr1""","""HAVANA""","""transcript""",11869,14409,""".""","""+""",""".""","""gene_id ""ENSG00000223972.5""; t…"
3,"""chr1""","""HAVANA""","""exon""",12010,12057,""".""","""+""",""".""","""gene_id ""ENSG00000223972.5""; t…"
4,"""chr1""","""HAVANA""","""transcript""",12010,13670,""".""","""+""",""".""","""gene_id ""ENSG00000223972.5""; t…"
…,…,…,…,…,…,…,…,…,…
3283857,"""chrY""","""HAVANA""","""transcript""",57212184,57214397,""".""","""-""",""".""","""gene_id ""ENSG00000227159.8_PAR…"
3283858,"""chrY""","""HAVANA""","""exon""",57213204,57213357,""".""","""-""",""".""","""gene_id ""ENSG00000227159.8_PAR…"
3283859,"""chrY""","""HAVANA""","""exon""",57213526,57213602,""".""","""-""",""".""","""gene_id ""ENSG00000227159.8_PAR…"


In [24]:
atts = (
    gtf_pl.select("row_nr", "attributes")
    
    .with_columns(
        attrs_list=pl.col("attributes").str.split("; ")
    )
    .explode("attrs_list")
    
    .with_columns(
        pl.col("attrs_list")
        .str.split_exact(" ", 1)
        .struct.rename_fields(["attribute", "value"])
        .alias("fields")
    ).unnest("fields")

    .with_columns(
        pl.col("value").str.strip_chars('"')
    )

    .pivot(
        index="row_nr",
        on="attribute",
        values="value",
        aggregate_function=pl.element().implode()
    )

    .with_columns(
        pl.col(pl.List).list.join(", ").str.strip_chars(";").replace("", None)
    )

    .with_columns(
        level = pl.col('level').cast(pl.Int32),
        exon_number = pl.col('exon_number').cast(pl.Int32),
    )
)

atts

row_nr,gene_id,transcript_id,gene_type,gene_name,transcript_type,transcript_name,exon_number,exon_id,level,transcript_support_level,hgnc_id,tag,havana_gene,havana_transcript,ont,protein_id,ccdsid
u64,str,str,str,str,str,str,i32,str,i32,str,str,str,str,str,str,str,str
0,"""ENSG00000223972.5""","""ENST00000456328.2""","""transcribed_unprocessed_pseudo…","""DDX11L1""","""processed_transcript""","""DDX11L1-202""",1,"""ENSE00002234944.1""",2,"""1""","""HGNC:37102""","""basic""","""OTTHUMG00000000961.2""","""OTTHUMT00000362751.1""""",null,null,null
1,"""ENSG00000223972.5""",null,"""transcribed_unprocessed_pseudo…","""DDX11L1""",null,null,null,null,2,null,"""HGNC:37102""",null,"""OTTHUMG00000000961.2""""",null,null,null,null
2,"""ENSG00000223972.5""","""ENST00000456328.2""","""transcribed_unprocessed_pseudo…","""DDX11L1""","""processed_transcript""","""DDX11L1-202""",null,null,2,"""1""","""HGNC:37102""","""basic""","""OTTHUMG00000000961.2""","""OTTHUMT00000362751.1""""",null,null,null
3,"""ENSG00000223972.5""","""ENST00000450305.2""","""transcribed_unprocessed_pseudo…","""DDX11L1""","""transcribed_unprocessed_pseudo…","""DDX11L1-201""",1,"""ENSE00001948541.1""",2,"""NA""","""HGNC:37102""","""basic, Ensembl_canonical""","""OTTHUMG00000000961.2""","""OTTHUMT00000002844.2""""","""PGO:0000005, PGO:0000019""",null,null
4,"""ENSG00000223972.5""","""ENST00000450305.2""","""transcribed_unprocessed_pseudo…","""DDX11L1""","""transcribed_unprocessed_pseudo…","""DDX11L1-201""",null,null,2,"""NA""","""HGNC:37102""","""basic, Ensembl_canonical""","""OTTHUMG00000000961.2""","""OTTHUMT00000002844.2""""","""PGO:0000005, PGO:0000019""",null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
3283857,"""ENSG00000227159.8_PAR_Y""","""ENST00000507418.6_PAR_Y""","""unprocessed_pseudogene""","""DDX11L16""","""unprocessed_pseudogene""","""DDX11L16-201""",null,null,2,"""NA""","""HGNC:37115""","""basic, Ensembl_canonical, PAR""","""OTTHUMG00000022678.1""","""OTTHUMT00000058841.1""""","""PGO:0000005""",null,null
3283858,"""ENSG00000227159.8_PAR_Y""","""ENST00000507418.6_PAR_Y""","""unprocessed_pseudogene""","""DDX11L16""","""unprocessed_pseudogene""","""DDX11L16-201""",4,"""ENSE00002036959.1""",2,"""NA""","""HGNC:37115""","""basic, Ensembl_canonical, PAR""","""OTTHUMG00000022678.1""","""OTTHUMT00000058841.1""""","""PGO:0000005""",null,null
3283859,"""ENSG00000227159.8_PAR_Y""","""ENST00000507418.6_PAR_Y""","""unprocessed_pseudogene""","""DDX11L16""","""unprocessed_pseudogene""","""DDX11L16-201""",3,"""ENSE00002021169.1""",2,"""NA""","""HGNC:37115""","""basic, Ensembl_canonical, PAR""","""OTTHUMG00000022678.1""","""OTTHUMT00000058841.1""""","""PGO:0000005""",null,null


In [25]:
gtf_genes = (
    gtf_pl
    .drop('attributes')
    .join(atts, on='row_nr')
    .with_columns(
        region = pl.col('gene_id').str.split('.').list.get(0),
        gene_start_pad = pl.col('Start') - 10_000,
        gene_end_pad = pl.col('End') + 10_000,
    )

    .filter(
        (pl.col('Feature') == 'gene') &
        (pl.col('region').is_in(bt_genes + olink_genes))
    )

    .select(['region', 'Chromosome', 'Start', 'End', 'gene_start_pad', 'gene_end_pad', 'Strand', 'gene_name', 'gene_type'])

    .rename({
        'Chromosome': 'chr',
        'Start': 'gene_start',
        'End': 'gene_end',
        'Strand': 'gene_strand',
    })
)

gtf_genes

region,chr,gene_start,gene_end,gene_start_pad,gene_end_pad,gene_strand,gene_name,gene_type
str,str,i64,i64,i64,i64,str,str,str
"""ENSG00000188157""","""chr1""",1020120,1056118,1010120,1066118,"""+""","""AGRN""","""protein_coding"""
"""ENSG00000186827""","""chr1""",1211340,1214153,1201340,1224153,"""-""","""TNFRSF4""","""protein_coding"""
"""ENSG00000224051""","""chr1""",1324756,1328896,1314756,1338896,"""+""","""CPTP""","""protein_coding"""
"""ENSG00000162576""","""chr1""",1352689,1361777,1342689,1371777,"""-""","""MXRA8""","""protein_coding"""
"""ENSG00000179403""","""chr1""",1434861,1442882,1424861,1452882,"""+""","""VWA1""","""protein_coding"""
…,…,…,…,…,…,…,…,…
"""ENSG00000198910""","""chrX""",153861514,153886173,153851514,153896173,"""-""","""L1CAM""","""protein_coding"""
"""ENSG00000102030""","""chrX""",153929225,153935080,153919225,153945080,"""-""","""NAA10""","""protein_coding"""
"""ENSG00000184216""","""chrX""",154010506,154019902,154000506,154029902,"""-""","""IRAK1""","""protein_coding"""


## Intersect with region files from the RAP

In [26]:
!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/wgs/all_region_files.parquet -o /home/dnanexus/data_dir/

Error: path "/home/dnanexus/data_dir/all_region_files.parquet" already exists
but -f/--overwrite was not set


In [27]:
reg_files = (
    pl.read_parquet('/home/dnanexus/data_dir/all_region_files.parquet')
    .drop('__index_level_0__')
    .with_columns(
        file_num = pl.col('file_name').str.split('.').list.get(0).str.slice(17)
    )
)
reg_files

file_name,chr,start,end,n_variants,file_num
str,str,i64,i64,i64,str
"""regions_ukb24310_c1_b11508_v1.…","""chr1""",230154171,230174165,7011,"""c1_b11508_v1"""
"""regions_ukb24310_c1_b1165_v1.t…","""chr1""",23300008,23319999,6784,"""c1_b1165_v1"""
"""regions_ukb24310_c1_b1104_v1.t…","""chr1""",22080001,22099999,7036,"""c1_b1104_v1"""
"""regions_ukb24310_c1_b7784_v1.t…","""chr1""",155676114,155696112,7291,"""c1_b7784_v1"""
"""regions_ukb24310_c1_b3160_v1.t…","""chr1""",63199036,63219025,6751,"""c1_b3160_v1"""
…,…,…,…,…,…
"""regions_ukb24310_c22_b552_v1.t…","""chr22""",11040002,11059997,5536,"""c22_b552_v1"""
"""regions_ukb24310_c22_b1866_v1.…","""chr22""",37319029,37339023,7265,"""c22_b1866_v1"""
"""regions_ukb24310_c22_b2160_v1.…","""chr22""",43199029,43219025,8328,"""c22_b2160_v1"""


In [28]:
# 1. Perform an inner join on 'chr'
# This matches genes to ALL files on the same chromosome first.
# We add a suffix to avoid column name collisions if both DFs have 'start'/'end' columns.
matches = (
    gtf_genes.join(reg_files, on="chr", suffix="_file")
    .filter(
        # 2. Filter for Interval Overlap
        # A gene overlaps a file if:
        # (Gene Start <= File End) AND (Gene End >= File Start)
        (pl.col("gene_start_pad") <= pl.col("end")) & 
        (pl.col("gene_end_pad") >= pl.col("start"))
    )
)

# 3. Extract the unique list of files to process
files_to_process = matches.select(["file_name", 'file_num']).unique()

# Optional: View the result
print(f"Found {files_to_process.height} files covering the target genes.")
# files_to_process.write_csv('/home/dnanexus/data_dir/genepheno_olink_bcf_file_list.tsv', separator='\t')
files_to_process

Found 19397 files covering the target genes.


file_name,file_num
str,str
"""regions_ukb24310_c14_b2601_v1.…","""c14_b2601_v1"""
"""regions_ukb24310_c4_b5479_v1.t…","""c4_b5479_v1"""
"""regions_ukb24310_c11_b6155_v1.…","""c11_b6155_v1"""
"""regions_ukb24310_c4_b9267_v1.t…","""c4_b9267_v1"""
"""regions_ukb24310_c3_b779_v1.tx…","""c3_b779_v1"""
…,…
"""regions_ukb24310_c19_b551_v1.t…","""c19_b551_v1"""
"""regions_ukb24310_c4_b1555_v1.t…","""c4_b1555_v1"""
"""regions_ukb24310_c12_b2232_v1.…","""c12_b2232_v1"""


In [30]:
files2download = sorted([f"project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/wgs/qced_maf1e-3.parquet/norm_qced_ukb24310_{fnum}.parquet" for fnum in files_to_process['file_num'].unique()])
files2download

['project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/wgs/qced_maf1e-3.parquet/norm_qced_ukb24310_c10_b1000_v1.parquet',
 'project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/wgs/qced_maf1e-3.parquet/norm_qced_ukb24310_c10_b1001_v1.parquet',
 'project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/wgs/qced_maf1e-3.parquet/norm_qced_ukb24310_c10_b1002_v1.parquet',
 'project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/wgs/qced_maf1e-3.parquet/norm_qced_ukb24310_c10_b1003_v1.parquet',
 'project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/wgs/qced_maf1e-3.parquet/norm_qced_ukb24310_c10_b1004_v1.parquet',
 'project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/wgs/qced_maf1e-3.parquet/norm_qced_ukb24310_c10_b1005_v1.parquet',
 'project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/wgs/qced_maf1e-3.parquet/norm_qced_ukb24310_c10_b1006_v1.parquet',
 'project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/wgs/qced_maf1e-3.parquet/norm_qced_ukb24310_c10_b1007_v1.parquet',
 'project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/wgs/q

In [ ]:
import subprocess
from concurrent.futures import ThreadPoolExecutor
from tqdm.notebook import tqdm  # Progress bar

# --- CONFIGURATION ---
OUTPUT_DIR = "/home/dnanexus/data_dir/bcf2parquet"      # Folder to save files to
MAX_WORKERS = 8                 # Number of parallel downloads (Don't go too high or you'll hit API limits)
# ---------------------

# Ensure output directory exists
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

def download_file(file_identifier):
    """
    Runs the dx download command for a single file.
    """
    try:
        # Construct the command
        # -f: Force overwrite (optional)
        # -o: Output directory
        cmd = ["dx", "download", file_identifier, "-o", OUTPUT_DIR, "-f"]
        
        # Run command and capture output
        result = subprocess.run(
            cmd, 
            capture_output=True, 
            text=True, 
            check=True
        )
        return True, file_identifier
    except subprocess.CalledProcessError as e:
        # Return False and the error message if it fails
        return False, f"{file_identifier}: {e.stderr}"

# Run the downloads in parallel
print(f"Starting download of {len(files2download)} files with {MAX_WORKERS} threads...")

failed_files = []

# ThreadPoolExecutor manages the pool of worker threads
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    # Map the function to the file list and wrap in tqdm for a progress bar
    results = list(tqdm(executor.map(download_file, files2download), total=len(files2download)))

# Process results
successful = sum(1 for status, _ in results if status)
for status, msg in results:
    if not status:
        failed_files.append(msg)

print(f"\nDownload Complete.")
print(f"Success: {successful}")
print(f"Failed:  {len(failed_files)}")

if failed_files:
    print("\nErrors:", failed_files[:5]) # Show first 5 errors

Starting download of 19397 files with 8 threads...


  0%|          | 0/19397 [00:00<?, ?it/s]